# Issue #10: Exploratory Data Analysis – Charging Demand Patterns

**Input:** `acn_clean_sessions.csv` and `acn_flagged_sessions.csv` (Issue #9)  
**Outputs:** `eda_tables/*.csv` (statistical summaries), `eda_key_findings.md` (numbers computed from the data)

This notebook is the *statistical* analysis: tables, tests and anomaly checks. The chart set is a separate issue.

**Reading rules used throughout**
- With ≈14,000 sessions, almost any difference is "statistically significant". Tests are reported with an **effect size**; judge importance from the effect size and the medians, not from p-values.
- **Data gaps:** runs of 7 or more consecutive days with zero sessions are treated as outages or missing data, not as zero demand. They are listed in Section 0b and excluded from daily statistics, per-day rates, trend tests and the plugged-in profile. Without this, a gap drags every average down and inflates every "variability" number.
- **Coverage:** the data is the main contiguous block of the collected sessions (Issue #9), not the full API history, and there is no full annual cycle. Partial months/weeks are excluded from trend statistics, and trends are checked against the number of active stations.
- **Policy regimes:** the site operator changed how sessions were handled during this period (Lee, Li & Low, *ACN-Data: Analysis and Applications of an Open EV Charging Dataset*, e-Energy 2019, Sec. 2.2, 3.1.2 and Fig. 2). Sessions without app input ("unclaimed", no `userID`) received a default energy allowance that differed by charger until 1 Sep 2018 and was 14 kWh over 12 h afterwards; from 1 Nov 2018 charging cost $0.12/kWh for claimed sessions and unclaimed sessions were stopped after 30 minutes. Each session is tagged with its regime (`regime`) and Section 7e compares the regimes. Pooled statistics mix regimes and should be read with that in mind.
- **Stay split:** short vs. long stays use the density valley between the two connection-time modes, computed the same way as in Issue #9.

In [1]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

try:
    from scipy import stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print("scipy not installed: statistical tests will be skipped (pip install scipy)")

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
os.makedirs("eda_tables", exist_ok=True)

def tab(name, frame, show=True):
    """Save a table to eda_tables/ and display it."""
    frame.to_csv(f"eda_tables/{name}.csv")
    if show:
        display(frame)
    return frame

LOCAL_TZ = "America/Los_Angeles"
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

df = pd.read_csv("../acn_clean_sessions.csv")
for c in ["connectionTime", "disconnectTime", "doneChargingTime"]:
    df[c] = pd.to_datetime(df[c], utc=True)
    df[c + "_loc"] = df[c].dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)

df["date"] = df["connectionTime_loc"].dt.normalize()
df["month"] = df["connectionTime_loc"].dt.to_period("M")
df["hour"] = df["connectionTime_loc"].dt.hour
df["dayofweek"] = df["connectionTime_loc"].dt.dayofweek
df["is_weekend"] = df["dayofweek"] >= 5
df["day_type"] = np.where(df["is_weekend"], "Weekend", "Weekday")
if "power_implausible" not in df.columns:
    df["power_implausible"] = False
df["done_ok"] = df["charging_h"].notna() & ~df["power_implausible"].astype(bool)
df["idle_h"] = np.where(df["done_ok"], (df["connected_h"] - df["charging_h"]).clip(lower=0), np.nan)
df["idle_share"] = df["idle_h"] / df["connected_h"]
# Short/long split at the density valley between the two connection-time modes (same method as Issue #9)
_h = df["connected_h"]
_counts, _edges = np.histogram(_h[(_h > 0) & (_h < 12)], bins=np.arange(0, 12.25, 0.25))
_smooth = pd.Series(_counts).rolling(5, center=True, min_periods=1).mean().values
_centres = (_edges[:-1] + _edges[1:]) / 2
_win = (_centres >= 1) & (_centres <= 10)
valley_h = float(_centres[_win][np.argmin(_smooth[_win])])
SHORT_STAY_H = round(valley_h * 2) / 2
SHORT_LABEL, LONG_LABEL = f"Short (<{SHORT_STAY_H:g} h)", f"Long (>={SHORT_STAY_H:g} h)"
df["stay"] = np.where(df["connected_h"] < SHORT_STAY_H, SHORT_LABEL, LONG_LABEL)

# Policy regimes (source: Lee, Li & Low 2019, ACN-Data paper, Fig. 2 and Sec. 3.1.2)
POLICY_DATES = [pd.Timestamp("2018-09-01"), pd.Timestamp("2018-11-01")]
REGIMES = ["R1 free, charger-specific defaults (to 31 Aug 2018)",
           "R2 free, 14 kWh default (1 Sep - 31 Oct 2018)",
           "R3 paid $0.12/kWh, unclaimed stopped after 30 min (from 1 Nov 2018)"]
df["regime"] = np.select([df["connectionTime_loc"] < POLICY_DATES[0], df["connectionTime_loc"] < POLICY_DATES[1]],
                         REGIMES[:2], default=REGIMES[2])
df["regime"] = pd.Categorical(df["regime"], categories=REGIMES, ordered=True)

# Long sessions (> 72 h) removed in Issue #9 still occupied a charger: add them back for occupancy only
FLAGGED_FILE = "../acn_flagged_sessions.csv"
long_sessions = pd.DataFrame(columns=["connectionTime_loc", "disconnectTime_loc"])
if os.path.exists(FLAGGED_FILE):
    _fl = pd.read_csv(FLAGGED_FILE)
    _fl = _fl[_fl["drop_reason"].astype(str).str.startswith("connected >")]
    for c in ["connectionTime", "disconnectTime"]:
        _fl[c + "_loc"] = pd.to_datetime(_fl[c], utc=True).dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)
    long_sessions = _fl[["connectionTime_loc", "disconnectTime_loc"]]
print(f"Stay split at {SHORT_STAY_H:g} h (valley ~{valley_h:.2f} h) | long sessions added back for occupancy: {len(long_sessions)}")

all_days = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
n_days = len(all_days)

# --- Gap detection: >= MIN_GAP_DAYS consecutive days with no sessions = outage / missing data ---
MIN_GAP_DAYS = 7
daily_counts_all = df.groupby("date").size().reindex(all_days, fill_value=0)
is_zero = daily_counts_all == 0
run_id = (is_zero != is_zero.shift()).cumsum()
run_len = is_zero.groupby(run_id).transform("sum")
gap_mask = is_zero & (run_len >= MIN_GAP_DAYS)
gap_dates = set(all_days[gap_mask.values])
gap_runs = (pd.DataFrame({"date": all_days[gap_mask.values], "run": run_id[gap_mask].values})
            .groupby("run")["date"].agg(gap_start="min", gap_end="max", days="count").reset_index(drop=True))

daily_counts = daily_counts_all[~gap_mask]                       # days with data ("service days")
dow_of_days = pd.Series(daily_counts.index.dayofweek, index=daily_counts.index)
n_service_days = len(daily_counts)
print(f"{len(df):,} sessions | {df['stationID'].nunique()} stations | {all_days.min().date()} to {all_days.max().date()} ({n_days} calendar days, {n_service_days} with data)")
print(f"Gap runs (>= {MIN_GAP_DAYS} consecutive zero-session days): {len(gap_runs)} covering {int(gap_mask.sum())} days")

Stay split at 6 h (valley ~6.12 h) | long sessions added back for occupancy: 21
14,279 sessions | 54 stations | 2018-04-25 to 2018-11-28 (218 calendar days, 218 with data)
Gap runs (>= 7 consecutive zero-session days): 0 covering 0 days


In [2]:
def describe(s, name=None):
    s = s.dropna()
    return pd.Series({
        "count": len(s), "mean": s.mean(), "std": s.std(), "cv": s.std() / s.mean() if s.mean() else np.nan,
        "min": s.min(), "p5": s.quantile(.05), "p25": s.quantile(.25), "median": s.median(),
        "p75": s.quantile(.75), "p95": s.quantile(.95), "max": s.max(),
        "skew": s.skew(), "kurtosis": s.kurt()}, name=name)

def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))

def compare(a, b, label_a, label_b):
    """Median comparison with Mann-Whitney U and a rank-biserial effect size (-1..1)."""
    a, b = a.dropna(), b.dropna()
    out = {"n_" + label_a: len(a), "n_" + label_b: len(b),
           "median_" + label_a: a.median(), "median_" + label_b: b.median(),
           "mean_" + label_a: a.mean(), "mean_" + label_b: b.mean()}
    if HAS_SCIPY and len(a) > 5 and len(b) > 5:
        u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        out["p_value"] = p
        out["effect_size_rb"] = 2 * u / (len(a) * len(b)) - 1     # >0: first group tends to be larger
    return pd.Series(out)

def effect_label(r):
    r = abs(r)
    return "negligible" if r < 0.1 else "small" if r < 0.3 else "medium" if r < 0.5 else "large"

## 0. Statistical summary of the main variables

In [3]:
summary_stats = pd.DataFrame({
    "kWh per session": describe(df["kWhDelivered"]),
    "Hours plugged in": describe(df["connected_h"]),
    "Hours charging": describe(df.loc[df["done_ok"], "charging_h"]),
    "Idle hours": describe(df["idle_h"]),
    "Idle share of stay": describe(df["idle_share"]),
    "Avg power (kW)": describe(df["avg_power_kW"]),
    "Sessions per day (days with data)": describe(daily_counts),
}).T
tab("00_summary_statistics", summary_stats)

,count,mean,std,cv,min,p5,p25,median,p75,p95,max,skew,kurtosis
kWh per session,"14,279.00",8.98,6.92,0.77,0.50,1.15,4.04,7.47,13.20,19.39,69.37,2.11,8.28
Hours plugged in,"14,279.00",5.78,5.23,0.90,0.09,0.51,2.01,4.71,8.72,12.52,71.00,3.72,29.57
Hours charging,"14,231.00",3.23,2.94,0.91,0.00,0.43,1.28,2.25,4.26,8.89,45.46,2.65,14.84
Idle hours,"14,231.00",2.55,4.37,1.71,0.00,0.00,0.00,0.69,4.07,9.03,65.09,5.17,48.51
Idle share of stay,"14,231.00",0.30,0.32,1.07,0.00,0.00,0.00,0.17,0.60,0.87,1.00,0.56,-1.24
Avg power (kW),"14,229.00",3.51,1.84,0.52,0.04,1.10,2.03,3.10,5.39,6.58,9.94,0.44,-1.00
Sessions per day (days with data),218.00,65.50,22.01,0.34,7.00,25.85,51.00,69.00,83.00,93.15,106.00,-0.49,-0.58


,count,mean,std,cv,min,p5,p25,median,p75,p95,max,skew,kurtosis
kWh per session,"14,279.00",8.98,6.92,0.77,0.50,1.15,4.04,7.47,13.20,19.39,69.37,2.11,8.28
Hours plugged in,"14,279.00",5.78,5.23,0.90,0.09,0.51,2.01,4.71,8.72,12.52,71.00,3.72,29.57
Hours charging,"14,231.00",3.23,2.94,0.91,0.00,0.43,1.28,2.25,4.26,8.89,45.46,2.65,14.84
Idle hours,"14,231.00",2.55,4.37,1.71,0.00,0.00,0.00,0.69,4.07,9.03,65.09,5.17,48.51
Idle share of stay,"14,231.00",0.30,0.32,1.07,0.00,0.00,0.00,0.17,0.60,0.87,1.00,0.56,-1.24
Avg power (kW),"14,229.00",3.51,1.84,0.52,0.04,1.10,2.03,3.10,5.39,6.58,9.94,0.44,-1.00
Sessions per day (days with data),218.00,65.50,22.01,0.34,7.00,25.85,51.00,69.00,83.00,93.15,106.00,-0.49,-0.58


### 0b. Data gaps (consecutive days with no sessions)

In [4]:
tab("00b_data_gaps", gap_runs)
print(f"{int(gap_mask.sum())} of {n_days} calendar days sit inside a gap and are excluded from daily statistics.")
print("Zero-session days OUTSIDE gaps (isolated closures):", int((daily_counts == 0).sum()))

,gap_start,gap_end,days


0 of 218 calendar days sit inside a gap and are excluded from daily statistics.
Zero-session days OUTSIDE gaps (isolated closures): 0


> A gap means the source recorded nothing for a week or more. It could be a real shutdown or a recording/collection outage, and this data cannot tell which. Either way it is not "demand was zero", so it must not enter averages. State the gap dates in the report.

> `skew` well above 0 means a long right tail (a few very large values pull the mean above the median); use medians for "typical" values. `cv` (std/mean) compares variability across variables measured in different units.

## 1. Charging-session frequency

In [5]:
by_type = pd.DataFrame({
    "All days": describe(daily_counts),
    "Weekdays": describe(daily_counts[dow_of_days < 5]),
    "Weekends": describe(daily_counts[dow_of_days >= 5]),
}).T
by_type["zero_session_days"] = [int((daily_counts == 0).sum()), int((daily_counts[dow_of_days < 5] == 0).sum()), int((daily_counts[dow_of_days >= 5] == 0).sum())]
by_type["dispersion_index"] = [daily_counts.var() / daily_counts.mean(),
                               daily_counts[dow_of_days < 5].var() / daily_counts[dow_of_days < 5].mean(),
                               daily_counts[dow_of_days >= 5].var() / daily_counts[dow_of_days >= 5].mean()]
tab("01a_daily_session_frequency", by_type)

,count,mean,std,cv,min,p5,p25,median,p75,p95,max,skew,kurtosis,zero_session_days,dispersion_index
All days,218.00,65.50,22.01,0.34,7.00,25.85,51.00,69.00,83.00,93.15,106.00,-0.49,-0.58,0,7.39
Weekdays,156.00,74.89,17.27,0.23,7.00,46.75,66.00,78.00,87.50,95.25,106.00,-1.25,2.50,0,3.98
Weekends,62.00,41.87,13.09,0.31,16.00,20.05,32.00,42.50,52.00,59.95,68.00,-0.11,-0.88,0,4.09


,count,mean,std,cv,min,p5,p25,median,p75,p95,max,skew,kurtosis,zero_session_days,dispersion_index
All days,218.00,65.50,22.01,0.34,7.00,25.85,51.00,69.00,83.00,93.15,106.00,-0.49,-0.58,0,7.39
Weekdays,156.00,74.89,17.27,0.23,7.00,46.75,66.00,78.00,87.50,95.25,106.00,-1.25,2.50,0,3.98
Weekends,62.00,41.87,13.09,0.31,16.00,20.05,32.00,42.50,52.00,59.95,68.00,-0.11,-0.88,0,4.09


> `dispersion_index` = variance / mean of daily counts. Near 1 means counts vary like pure random arrivals; well above 1 means days differ systematically (weekday effects, holidays, growth).

In [6]:
# Weekly and monthly frequency, complete periods only (partial first/last periods would bias the numbers)
week_start = (df["date"] - pd.to_timedelta(df["dayofweek"], unit="D"))
weekly = df.groupby(week_start).size()
def week_clean(ws):
    return ws >= all_days.min() and ws + pd.Timedelta(days=6) <= all_days.max() and not any((ws + pd.Timedelta(days=i)) in gap_dates for i in range(7))
complete_weeks = weekly[[week_clean(ws) for ws in weekly.index]]

monthly = df.groupby("month").agg(sessions=("_id", "count"), active_stations=("stationID", "nunique"),
                                  total_kwh=("kWhDelivered", "sum"), median_kwh=("kWhDelivered", "median"),
                                  median_hours=("connected_h", "median"))
svc_per_month = pd.Series(1, index=daily_counts.index).groupby(daily_counts.index.to_period("M")).sum()
monthly["service_days"] = svc_per_month.reindex(monthly.index).fillna(0).astype(int)
monthly["days_in_month"] = monthly.index.days_in_month
monthly["sessions_per_service_day"] = monthly["sessions"] / monthly["service_days"].replace(0, np.nan)
monthly["sessions_per_station_per_service_day"] = monthly["sessions_per_service_day"] / monthly["active_stations"]
# usable for trend: month lies fully inside the data range and at least 80% of its days have data
inside = ~monthly.index.isin([monthly.index[0], monthly.index[-1]])
monthly["complete_month"] = inside & (monthly["service_days"] >= 0.8 * monthly["days_in_month"])
tab("01b_monthly_frequency", monthly)

missing_months = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M").difference(monthly.index)
if len(missing_months):
    print("Months with NO sessions at all:", [str(m) for m in missing_months])
print(f"Complete weeks (no gap days): {len(complete_weeks)} | mean {complete_weeks.mean():.0f} sessions/week | std {complete_weeks.std():.0f} | min {complete_weeks.min()} | max {complete_weeks.max()}")

,sessions,active_stations,total_kwh,median_kwh,median_hours,service_days,days_in_month,sessions_per_service_day,sessions_per_station_per_service_day,complete_month
month,,,,,,,,,,
2018-04,268,43,"2,388.36",6.53,4.82,6,30,44.67,1.04,False
2018-05,1774,51,"15,681.59",7.32,5.48,31,31,57.23,1.12,True
2018-06,1843,52,"16,884.73",7.89,5.22,30,30,61.43,1.18,True
2018-07,2049,53,"18,950.34",7.47,4.53,31,31,66.10,1.25,True
2018-08,2427,53,"20,583.05",7.33,4.31,31,31,78.29,1.48,True
2018-09,2315,54,"20,615.23",7.92,4.11,30,30,77.17,1.43,True
2018-10,2420,54,"22,524.17",7.91,4.57,31,31,78.06,1.45,True
2018-11,1183,53,"10,544.04",6.34,5.91,28,30,42.25,0.80,False


Complete weeks (no gap days): 30 | mean 464 sessions/week | std 89 | min 210 | max 575


## 2. Energy consumption patterns

In [7]:
BLOCKS = [(0, 6, "Night 00–06"), (6, 10, "Morning 06–10"), (10, 16, "Midday 10–16"), (16, 20, "Evening 16–20"), (20, 24, "Late 20–24")]
df["time_block"] = ""
for lo, hi, name in BLOCKS:
    df.loc[(df["hour"] >= lo) & (df["hour"] < hi), "time_block"] = name

def group_energy(by):
    g = df.groupby(by).agg(sessions=("_id", "count"), total_kwh=("kWhDelivered", "sum"),
                           mean_kwh=("kWhDelivered", "mean"), median_kwh=("kWhDelivered", "median"),
                           p95_kwh=("kWhDelivered", lambda s: s.quantile(.95)))
    g["share_of_sessions"] = g["sessions"] / g["sessions"].sum()
    g["share_of_energy"] = g["total_kwh"] / g["total_kwh"].sum()
    return g

tab("02a_energy_by_day_type", group_energy("day_type"))
blk_tab = group_energy("time_block").reindex([b[2] for b in BLOCKS])
blk_tab["block_hours"] = [hi - lo for lo, hi, _ in BLOCKS]
blk_tab["sessions_per_day_per_block_hour"] = blk_tab["sessions"] / blk_tab["block_hours"] / n_service_days   # blocks differ in length
tab("02b_energy_by_time_block", blk_tab)
tab("02c_energy_by_stay_type", group_energy("stay"))

,sessions,total_kwh,mean_kwh,median_kwh,p95_kwh,share_of_sessions,share_of_energy
day_type,,,,,,,
Weekday,11683,"103,294.16",8.84,7.40,18.88,0.82,0.81
Weekend,2596,"24,877.36",9.58,8.00,23.74,0.18,0.19


,sessions,total_kwh,mean_kwh,median_kwh,p95_kwh,share_of_sessions,share_of_energy,block_hours,sessions_per_day_per_block_hour
time_block,,,,,,,,,
Night 00–06,413,"4,320.50",10.46,10.97,34.46,0.03,0.03,6,0.32
Morning 06–10,6010,"54,878.17",9.13,7.71,19.57,0.42,0.43,4,6.89
Midday 10–16,3657,"30,150.78",8.24,7.10,17.12,0.26,0.24,6,2.80
Evening 16–20,2920,"26,732.32",9.15,7.85,18.95,0.20,0.21,4,3.35
Late 20–24,1279,"12,089.75",9.45,6.88,27.12,0.09,0.09,4,1.47


,sessions,total_kwh,mean_kwh,median_kwh,p95_kwh,share_of_sessions,share_of_energy
stay,,,,,,,
Long (>=6 h),6120,"68,055.91",11.12,10.44,26.33,0.43,0.53
Short (<6 h),8159,"60,115.61",7.37,6.21,14.72,0.57,0.47


,sessions,total_kwh,mean_kwh,median_kwh,p95_kwh,share_of_sessions,share_of_energy
stay,,,,,,,
Long (>=6 h),6120,"68,055.91",11.12,10.44,26.33,0.43,0.53
Short (<6 h),8159,"60,115.61",7.37,6.21,14.72,0.57,0.47


In [8]:
# Is energy per session different on weekdays vs weekends? Effect size matters more than the p-value.
wk_kwh = compare(df.loc[~df.is_weekend, "kWhDelivered"], df.loc[df.is_weekend, "kWhDelivered"], "weekday", "weekend")
tab("02d_test_weekday_vs_weekend_kwh", wk_kwh.to_frame("value"))
if "effect_size_rb" in wk_kwh:
    print("Effect size:", round(wk_kwh["effect_size_rb"], 3), "->", effect_label(wk_kwh["effect_size_rb"]))

if HAS_SCIPY:
    groups = [g["kWhDelivered"].values for _, g in df.groupby("time_block")]
    h, p = stats.kruskal(*groups)
    eps2 = h / (len(df) - 1)                      # epsilon-squared effect size
    print(f"Kruskal-Wallis across time blocks: H={h:.1f}, p={p:.2g}, epsilon^2={eps2:.3f}")

,value
n_weekday,"11,683.00"
n_weekend,"2,596.00"
median_weekday,7.40
median_weekend,8.00
mean_weekday,8.84
mean_weekend,9.58
p_value,0.06
effect_size_rb,-0.02


Effect size: -0.024 -> negligible
Kruskal-Wallis across time blocks: H=63.7, p=4.9e-13, epsilon^2=0.004


In [9]:
# Concentration of energy: how much of the total comes from the biggest sessions?
s = df["kWhDelivered"].sort_values(ascending=False)
conc = pd.DataFrame({
    "share_of_sessions": [0.01, 0.05, 0.10, 0.25, 0.50],
})
conc["share_of_energy"] = [s.head(int(len(s) * q)).sum() / s.sum() for q in conc["share_of_sessions"]]
tab("02e_energy_concentration", conc.set_index("share_of_sessions"))

,share_of_energy
share_of_sessions,
0.01,0.05
0.05,0.16
0.10,0.26
0.25,0.49
0.50,0.78


,share_of_energy
share_of_sessions,
0.01,0.05
0.05,0.16
0.10,0.26
0.25,0.49
0.50,0.78


## 3. Charging duration

In [10]:
bands = pd.cut(df["connected_h"], [0, 1, 3, 6, 9, 12, 24, np.inf],
               labels=["<1 h", "1–3 h", "3–6 h", "6–9 h", "9–12 h", "12–24 h", ">24 h"], right=False)
dur = df.groupby(bands, observed=True).agg(sessions=("_id", "count"), median_kwh=("kWhDelivered", "median"),
                                          median_idle_share=("idle_share", "median"),
                                          median_arrival_hour=("hour", "median"))
dur["share_of_sessions"] = dur["sessions"] / dur["sessions"].sum()
tab("03a_duration_bands", dur)

,sessions,median_kwh,median_idle_share,median_arrival_hour,share_of_sessions
connected_h,,,,,
<1 h,1542,2.14,0.00,16.00,0.11
1–3 h,3626,6.56,0.00,15.00,0.25
3–6 h,2991,9.88,0.20,12.00,0.21
6–9 h,2996,9.35,0.48,9.00,0.21
9–12 h,2286,10.60,0.59,8.00,0.16
12–24 h,710,13.27,0.69,17.00,0.05
>24 h,128,12.66,0.87,15.00,0.01


,sessions,median_kwh,median_idle_share,median_arrival_hour,share_of_sessions
connected_h,,,,,
<1 h,1542,2.14,0.00,16.00,0.11
1–3 h,3626,6.56,0.00,15.00,0.25
3–6 h,2991,9.88,0.20,12.00,0.21
6–9 h,2996,9.35,0.48,9.00,0.21
9–12 h,2286,10.60,0.59,8.00,0.16
12–24 h,710,13.27,0.69,17.00,0.05
>24 h,128,12.66,0.87,15.00,0.01


In [11]:
ok = df[df["done_ok"]]
dur_tot = pd.Series({
    "plugged-in hours (trusted-done sessions)": ok["connected_h"].sum(),
    "charging hours": ok["charging_h"].sum(),
    "idle hours": ok["idle_h"].sum(),
})
dur_tot["idle share of plugged-in hours"] = dur_tot["idle hours"] / dur_tot["plugged-in hours (trusted-done sessions)"]
tab("03b_charging_vs_idle_hours", dur_tot.to_frame("value"))

if HAS_SCIPY:
    r1, p1 = stats.spearmanr(df["kWhDelivered"], df["connected_h"])
    r2, p2 = stats.spearmanr(ok["kWhDelivered"], ok["charging_h"])
    r3, p3 = stats.spearmanr(df["hour"], df["connected_h"])
    corr = pd.DataFrame({"spearman_rho": [r1, r2, r3], "p_value": [p1, p2, p3]},
                        index=["kWh vs hours plugged in", "kWh vs hours charging", "arrival hour vs hours plugged in"])
    tab("03c_duration_correlations", corr)

,value
plugged-in hours (trusted-done sessions),"82,195.16"
charging hours,"45,927.32"
idle hours,"36,279.10"
idle share of plugged-in hours,0.44


,spearman_rho,p_value
kWh vs hours plugged in,0.42,0.00
kWh vs hours charging,0.73,0.00
arrival hour vs hours plugged in,-0.37,0.00


> Spearman rank correlation (robust to the heavy tails here). |rho| < 0.3 is weak, 0.3–0.6 moderate, > 0.6 strong. A weak kWh-vs-time-plugged-in link means long stays are mostly not about needing more energy — relevant to idle time and scheduling.

## 4. Demand by time of day

In [12]:
hourly = df.groupby("hour").agg(sessions=("_id", "count"), total_kwh=("kWhDelivered", "sum"),
                                median_kwh=("kWhDelivered", "median"), median_hours=("connected_h", "median"))
hourly = hourly.reindex(range(24), fill_value=0)
hourly["share_of_arrivals"] = hourly["sessions"] / hourly["sessions"].sum()
hourly["share_of_energy"] = hourly["total_kwh"] / hourly["total_kwh"].sum()
hourly["weekday_arrivals_per_day"] = df[~df.is_weekend].groupby("hour").size().reindex(range(24), fill_value=0) / int((dow_of_days < 5).sum())
hourly["weekend_arrivals_per_day"] = df[df.is_weekend].groupby("hour").size().reindex(range(24), fill_value=0) / int((dow_of_days >= 5).sum())
tab("04a_hourly_demand", hourly)

,sessions,total_kwh,median_kwh,median_hours,share_of_arrivals,share_of_energy,weekday_arrivals_per_day,weekend_arrivals_per_day
hour,,,,,,,,
0,92,906.95,6.21,4.25,0.01,0.01,0.35,0.61
1,58,632.08,6.36,3.48,0.00,0.00,0.22,0.39
2,23,338.26,8.35,3.26,0.00,0.00,0.10,0.13
3,23,163.24,6.16,2.92,0.00,0.00,0.09,0.15
4,136,"1,451.67",11.82,8.40,0.01,0.01,0.80,0.18
5,81,828.30,5.39,1.76,0.01,0.01,0.46,0.15
6,436,"4,066.37",8.58,8.63,0.03,0.03,2.62,0.44
7,1115,"11,048.88",9.41,8.84,0.08,0.09,6.37,1.95
8,2660,"24,024.94",7.03,8.36,0.19,0.19,16.29,1.92


,sessions,total_kwh,median_kwh,median_hours,share_of_arrivals,share_of_energy,weekday_arrivals_per_day,weekend_arrivals_per_day
hour,,,,,,,,
0,92,906.95,6.21,4.25,0.01,0.01,0.35,0.61
1,58,632.08,6.36,3.48,0.00,0.00,0.22,0.39
2,23,338.26,8.35,3.26,0.00,0.00,0.10,0.13
3,23,163.24,6.16,2.92,0.00,0.00,0.09,0.15
4,136,"1,451.67",11.82,8.40,0.01,0.01,0.80,0.18
5,81,828.30,5.39,1.76,0.01,0.01,0.46,0.15
6,436,"4,066.37",8.58,8.63,0.03,0.03,2.62,0.44
7,1115,"11,048.88",9.41,8.84,0.08,0.09,6.37,1.95
8,2660,"24,024.94",7.03,8.36,0.19,0.19,16.29,1.92


In [13]:
# Concurrency: vehicles plugged in at the same time (15-minute resolution).
# Includes the > 72 h sessions removed in Issue #9, because they occupied a charger.
occ = pd.concat([df[["connectionTime_loc", "disconnectTime_loc"]], long_sessions], ignore_index=True)
start = occ["connectionTime_loc"].dt.floor("15min")
end = occ["disconnectTime_loc"].dt.floor("15min")
events = pd.concat([pd.Series(1, index=start), pd.Series(-1, index=end)]).groupby(level=0).sum()
grid = pd.date_range(events.index.min(), events.index.max(), freq="15min")
plugged = events.reindex(grid, fill_value=0).cumsum()

pl = pd.DataFrame({"plugged": plugged})
pl["hour"] = pl.index.hour
pl["weekend"] = pl.index.dayofweek >= 5
pl = pl[~pl.index.normalize().isin(list(gap_dates))]          # exclude gap days: a gap would pull every average toward zero
conc_h = pl.groupby(["weekend", "hour"])["plugged"].agg(["mean", "max"]).unstack(0)
conc_h.columns = [f"{'weekend' if w else 'weekday'}_{m}_plugged_in" for m, w in conc_h.columns]
tab("04b_concurrency_by_hour", conc_h)

wd_mean = pl[~pl.weekend].groupby("hour")["plugged"].mean()
peak_to_avg = wd_mean.max() / wd_mean.mean()
print(f"Weekday: peak hourly mean {wd_mean.max():.1f} vehicles at {int(wd_mean.idxmax())}:00 | average over the day {wd_mean.mean():.1f} | "
      f"peak-to-average {peak_to_avg:.2f} | largest simultaneous count ever {int(plugged.max())}")

# Capacity pressure: how often are (almost) all chargers occupied?
N_STATIONS = df["stationID"].nunique()
pl["occupancy"] = pl["plugged"] / N_STATIONS
wd = pl[~pl.weekend]
wd_days = wd.index.normalize().nunique()
cap = pd.Series({
    "stations": N_STATIONS,
    "largest simultaneous count": int(plugged.max()),
    "peak occupancy (largest count / stations)": plugged.max() / N_STATIONS,
    "weekday 15-min slots with >= 90% of stations occupied": int((wd["occupancy"] >= 0.9).sum()),
    "weekday days with at least one such slot": int(wd.loc[wd["occupancy"] >= 0.9].index.normalize().nunique()),
    "share of weekdays with at least one such slot": wd.loc[wd["occupancy"] >= 0.9].index.normalize().nunique() / wd_days if wd_days else np.nan,
    "weekday mean occupancy 10:00-15:00": wd.loc[(wd.hour >= 10) & (wd.hour < 15), "occupancy"].mean(),
})
tab("04c_capacity_pressure", cap.to_frame("value"))

,weekday_mean_plugged_in,weekend_mean_plugged_in,weekday_max_plugged_in,weekend_max_plugged_in
hour,,,,
0,5.86,7.06,12,14
1,5.82,7.01,13,13
2,5.74,7.01,13,13
3,5.54,6.94,13,13
4,5.57,6.97,13,13
5,6.06,6.91,13,13
6,6.31,6.87,13,13
7,9.95,7.63,23,17
8,20.62,8.90,45,19


Weekday: peak hourly mean 38.5 vehicles at 11:00 | average over the day 18.9 | peak-to-average 2.03 | largest simultaneous count ever 51


,value
stations,54.00
largest simultaneous count,51.00
peak occupancy (largest count / stations),0.94
weekday 15-min slots with >= 90% of stations occupied,40.00
weekday days with at least one such slot,5.00
share of weekdays with at least one such slot,0.03
weekday mean occupancy 10:00-15:00,0.71


,value
stations,54.00
largest simultaneous count,51.00
peak occupancy (largest count / stations),0.94
weekday 15-min slots with >= 90% of stations occupied,40.00
weekday days with at least one such slot,5.00
share of weekdays with at least one such slot,0.03
weekday mean occupancy 10:00-15:00,0.71


## 5. Demand by day of week

In [14]:
n_each = dow_of_days.value_counts().reindex(range(7))
dow = df.groupby("dayofweek").agg(sessions=("_id", "count"), total_kwh=("kWhDelivered", "sum"),
                                  median_kwh=("kWhDelivered", "median"), median_hours=("connected_h", "median"))
dow["days_in_data"] = n_each
dow["sessions_per_day"] = dow["sessions"] / dow["days_in_data"]
dow["kwh_per_day"] = dow["total_kwh"] / dow["days_in_data"]
dow.index = DOW
tab("05a_day_of_week", dow)

if HAS_SCIPY:
    by_dow = [daily_counts[dow_of_days == k].values for k in range(7)]
    h, p = stats.kruskal(*by_dow)
    print(f"Kruskal-Wallis on daily session counts across weekdays: H={h:.1f}, p={p:.2g}, epsilon^2={h/(n_service_days-1):.3f}")
wdc = compare(daily_counts[dow_of_days < 5], daily_counts[dow_of_days >= 5], "weekday", "weekend")
tab("05b_test_weekday_vs_weekend_daily_sessions", wdc.to_frame("value"))

,sessions,total_kwh,median_kwh,median_hours,days_in_data,sessions_per_day,kwh_per_day
Mon,2261,"21,390.13",8.20,5.46,31,72.94,690.00
Tue,2402,"21,232.13",7.68,5.87,31,77.48,684.91
Wed,2372,"20,228.16",6.95,5.71,32,74.12,632.13
Thu,2347,"19,704.37",7.19,5.49,31,75.71,635.62
Fri,2301,"20,739.36",7.34,5.53,31,74.23,669.01
Sat,1310,"12,752.49",8.09,2.38,31,42.26,411.37
Sun,1286,"12,124.87",7.92,2.37,31,41.48,391.12


Kruskal-Wallis on daily session counts across weekdays: H=100.2, p=2.2e-19, epsilon^2=0.462


,value
n_weekday,156.00
n_weekend,62.00
median_weekday,78.00
median_weekend,42.50
mean_weekday,74.89
mean_weekend,41.87
p_value,0.00
effect_size_rb,0.87


,value
n_weekday,156.00
n_weekend,62.00
median_weekday,78.00
median_weekend,42.50
mean_weekday,74.89
mean_weekend,41.87
p_value,0.00
effect_size_rb,0.87


## 6. Station / site-level usage

In [15]:
n_sites = df["siteID"].nunique() if "siteID" in df.columns else 1      # siteID/clusterID were dropped in Issue #9 as constant
print("Sites:", n_sites, "| stations:", df["stationID"].nunique(), "(only one site, so site-level comparison is not possible)")

stn = df.groupby("stationID").agg(sessions=("_id", "count"), total_kwh=("kWhDelivered", "sum"),
                                  median_kwh=("kWhDelivered", "median"), median_hours=("connected_h", "median"),
                                  first_session=("date", "min"), last_session=("date", "max"))
stn["observed_days"] = (stn["last_session"] - stn["first_session"]).dt.days + 1
stn["sessions_per_observed_day"] = stn["sessions"] / stn["observed_days"]
stn = stn.sort_values("sessions", ascending=False)
tab("06a_station_usage", stn, show=False)
display(stn.head(10)); display(stn.tail(5))

Sites: 1 | stations: 54 (only one site, so site-level comparison is not possible)


,sessions,total_kwh,median_kwh,median_hours,first_session,last_session,observed_days,sessions_per_observed_day
stationID,,,,,,,,
2-39-139-28,654,"5,562.69",6.79,2.30,2018-04-25,2018-11-28,218,3.00
2-39-131-30,534,"4,880.82",7.94,2.87,2018-04-26,2018-11-28,217,2.46
2-39-129-17,508,"4,445.58",7.00,2.73,2018-04-30,2018-11-28,213,2.38
2-39-127-19,499,"4,183.03",6.72,2.73,2018-04-25,2018-11-27,217,2.30
2-39-138-29,479,"4,078.26",6.33,2.46,2018-04-25,2018-11-26,216,2.22
2-39-125-21,476,"3,748.35",6.61,2.43,2018-04-25,2018-11-28,218,2.18
2-39-89-25,446,"3,887.70",6.18,3.37,2018-04-25,2018-11-28,218,2.05
2-39-123-23,429,"3,438.43",6.51,3.12,2018-04-25,2018-11-28,218,1.97
2-39-130-31,422,"3,473.37",6.77,2.19,2018-04-26,2018-11-28,217,1.94


,sessions,total_kwh,median_kwh,median_hours,first_session,last_session,observed_days,sessions_per_observed_day
stationID,,,,,,,,
2-39-129-563,81,658.64,7.66,6.97,2018-05-03,2018-11-06,188,0.43
2-39-125-559,66,524.32,7.88,5.84,2018-04-25,2018-11-12,202,0.33
2-39-83-387,62,387.88,4.56,2.29,2018-06-03,2018-11-22,173,0.36
2-39-82-384,8,44.47,5.39,1.93,2018-07-18,2018-11-27,133,0.06
2-39-82-385,4,18.64,3.54,2.58,2018-09-03,2018-10-18,46,0.09


> `observed_days` runs from a station's first to last observed session. It is a proxy for when it was in service: a station that was installed early but rarely used looks the same as a late installation. Normalising by it (`sessions_per_observed_day`) removes most of the rollout effect but not all of it.

In [16]:
share = stn["sessions"].cumsum() / stn["sessions"].sum()
top10 = min(10, len(stn))
q1, q3 = stn["sessions_per_observed_day"].quantile([.25, .75]); iqr = q3 - q1
low_f, high_f = q1 - 1.5 * iqr, q3 + 1.5 * iqr
under = stn[stn["sessions_per_observed_day"] < low_f]; over = stn[stn["sessions_per_observed_day"] > high_f]

conc_stats = pd.Series({
    "stations": len(stn),
    "gini_sessions (0 = even, 1 = one station only)": gini(stn["sessions"]),
    "gini_sessions_per_observed_day": gini(stn["sessions_per_observed_day"]),
    f"top_{top10}_stations_share_of_sessions": share.iloc[top10 - 1],
    "even_share_for_same_number_of_stations": top10 / len(stn),
    "stations_first_seen_in_first_month": int((stn["first_session"] <= all_days.min() + pd.Timedelta(days=30)).sum()),
    "under-used stations (below Tukey fence, per observed day)": len(under),
    "over-used stations (above Tukey fence, per observed day)": len(over),
})
tab("06b_station_concentration", conc_stats.to_frame("value"))
if len(under): display(under[["sessions", "observed_days", "sessions_per_observed_day"]])
if len(over): display(over[["sessions", "observed_days", "sessions_per_observed_day"]])

,value
stations,54.00
"gini_sessions (0 = even, 1 = one station only)",0.31
gini_sessions_per_observed_day,0.30
top_10_stations_share_of_sessions,0.34
even_share_for_same_number_of_stations,0.19
stations_first_seen_in_first_month,51.00
"under-used stations (below Tukey fence, per observed day)",0.00
"over-used stations (above Tukey fence, per observed day)",1.00


,sessions,observed_days,sessions_per_observed_day
stationID,,,
2-39-139-28,654,218,3.00


## 7. Trends and anomalies

### 7a. Trend (complete months only, normalised by active stations)

In [17]:
full = monthly[monthly["complete_month"]].copy()
full["month_index"] = range(len(full))
tab("07a_complete_months", full[["sessions", "service_days", "active_stations", "sessions_per_service_day", "sessions_per_station_per_service_day", "median_kwh", "median_hours"]])

trend = {}
if HAS_SCIPY and len(full) >= 5:
    for col in ["sessions_per_service_day", "sessions_per_station_per_service_day", "median_kwh", "median_hours"]:
        rho, p = stats.spearmanr(full["month_index"], full[col])
        trend[col] = {"spearman_rho_vs_time": rho, "p_value": p}
    tab("07b_trend_tests", pd.DataFrame(trend).T)
    print(f"Based on only {len(full)} complete months: indicative at best.")

,sessions,service_days,active_stations,sessions_per_service_day,sessions_per_station_per_service_day,median_kwh,median_hours
month,,,,,,,
2018-05,1774,31,51,57.23,1.12,7.32,5.48
2018-06,1843,30,52,61.43,1.18,7.89,5.22
2018-07,2049,31,53,66.10,1.25,7.47,4.53
2018-08,2427,31,53,78.29,1.48,7.33,4.31
2018-09,2315,30,54,77.17,1.43,7.92,4.11
2018-10,2420,31,54,78.06,1.45,7.91,4.57


,spearman_rho_vs_time,p_value
sessions_per_service_day,0.83,0.04
sessions_per_station_per_service_day,0.83,0.04
median_kwh,0.71,0.11
median_hours,-0.66,0.16


Based on only 6 complete months: indicative at best.


> Everything here is per *day with data*, so a gap month is comparable with a full month. Compare `sessions_per_service_day` with `sessions_per_station_per_service_day`: if sessions rise but sessions per station do not, the growth is rollout, not behavior. With so few months, no trend should be called significant on the p-value alone.

### 7a-2. Last 16 weeks of data (or before the first gap): weekly weekday volume, spike share and share with `userID`

In [18]:
cut = gap_runs["gap_start"].min() if len(gap_runs) else daily_counts.index.max() + pd.Timedelta(days=1)
w = daily_counts[(daily_counts.index < cut) & (daily_counts.index >= cut - pd.Timedelta(weeks=16)) & (daily_counts.index.dayofweek < 5)]
wk_recent = w.resample("W").mean().to_frame("weekday_sessions_per_day")
_ix = df.set_index("connectionTime_loc")
wk_recent["spike_share"] = _ix["kWhDelivered"].between(13, 14, inclusive="left").resample("W").mean().reindex(wk_recent.index)
wk_recent["share_with_userID"] = _ix["has_user_id"].astype(bool).resample("W").mean().reindex(wk_recent.index)
wk_recent["policy_change_this_week"] = [any((d - pd.Timedelta(days=6) <= p <= d) for p in POLICY_DATES) for d in wk_recent.index]
tab("07j_last_weeks", wk_recent)

,weekday_sessions_per_day,spike_share,share_with_userID,policy_change_this_week
2018-08-12,89.00,0.23,0.02,False
2018-08-19,87.60,0.21,0.02,False
2018-08-26,85.40,0.24,0.03,False
2018-09-02,91.40,0.20,0.02,True
2018-09-09,80.60,0.27,0.05,False
2018-09-16,93.40,0.20,0.08,False
2018-09-23,91.80,0.23,0.06,False
2018-09-30,93.20,0.15,0.13,False
2018-10-07,91.00,0.17,0.15,False
2018-10-14,86.00,0.19,0.20,False


,weekday_sessions_per_day,spike_share,share_with_userID,policy_change_this_week
2018-08-12,89.00,0.23,0.02,False
2018-08-19,87.60,0.21,0.02,False
2018-08-26,85.40,0.24,0.03,False
2018-09-02,91.40,0.20,0.02,True
2018-09-09,80.60,0.27,0.05,False
2018-09-16,93.40,0.20,0.08,False
2018-09-23,91.80,0.23,0.06,False
2018-09-30,93.20,0.15,0.13,False
2018-10-07,91.00,0.17,0.15,False
2018-10-14,86.00,0.19,0.20,False


> A **sudden step** in weekday volume, spike share or `userID` share in a specific week points to an external change (pricing, policy, a software or recording change); weeks containing a documented policy date are marked. A **gradual slide** points to behaviour or to the recording system. Say what the table shows; Section 7e tests the documented changes directly.

### 7b. Anomalous days (relative to the same weekday in nearby weeks)

> Days just after 1 Nov 2018 may be flagged because the ±2-week baseline still contains free-charging weeks; check the `regime` of flagged days before calling them anomalies.

In [19]:
# Baseline = median of the same weekday over +/-2 weeks. This accounts for weekday effects and slow growth.
base = pd.Series(index=daily_counts.index, dtype=float)
for k in range(7):
    s = daily_counts[dow_of_days == k]
    base.loc[s.index] = s.rolling(5, center=True, min_periods=3).median()
dev = (daily_counts - base) / base.replace(0, np.nan)
mad = (dev - dev.median()).abs().median()
robust_z = 0.6745 * (dev - dev.median()) / mad if mad else dev * np.nan

try:
    from pandas.tseries.holiday import USFederalHolidayCalendar
    hol = USFederalHolidayCalendar().holidays(start=all_days.min(), end=all_days.max())
except Exception:
    hol = pd.DatetimeIndex([])

anom = pd.DataFrame({"sessions": daily_counts, "baseline": base, "pct_vs_baseline": dev * 100, "robust_z": robust_z})
anom["weekday"] = [DOW[d] for d in anom.index.dayofweek]
anom["us_federal_holiday"] = anom.index.isin(hol)
anom["after_paid_start"] = anom.index >= POLICY_DATES[1]
flag = anom[anom["robust_z"].abs() > 3.5].sort_values("robust_z", key=abs, ascending=False)
tab("07c_anomalous_days", flag)
print(f"{len(flag)} anomalous days of {len(anom)} | of these, on a US federal holiday: {int(flag['us_federal_holiday'].sum())}")

,sessions,baseline,pct_vs_baseline,robust_z,weekday,us_federal_holiday,after_paid_start
2018-11-23,7,54.00,-87.04,-14.53,Fri,False,True
2018-11-22,9,58.00,-84.48,-14.10,Thu,True,True
2018-05-28,20,66.00,-69.70,-11.63,Mon,True,False
2018-07-04,24,72.00,-66.67,-11.13,Wed,True,False
2018-09-03,34,90.00,-62.22,-10.39,Mon,True,False
2018-11-18,33,23.00,43.48,7.26,Sun,False,True
2018-11-11,16,26.00,-38.46,-6.42,Sun,False,True
2018-11-10,31,23.00,34.78,5.81,Sat,False,True
2018-08-11,38,58.00,-34.48,-5.76,Sat,False,False
2018-08-12,40,59.00,-32.20,-5.38,Sun,False,False


25 anomalous days of 218 | of these, on a US federal holiday: 4


### 7c. Anomalies in session values

In [20]:
e = df["kWhDelivered"]
q1, q3 = e.quantile([.25, .75]); fence = q3 + 1.5 * (q3 - q1)
anom_vals = pd.Series({
    "kWh upper Tukey fence": fence,
    "sessions above fence": int((e > fence).sum()),
    "share above fence": float((e > fence).mean()),
    "sessions plugged in > 24 h": int((df["connected_h"] > 24).sum()),
    "sessions plugged in < 15 min": int((df["connected_h"] < 0.25).sum()),
    "arrivals between 00:00 and 05:00": int((df["hour"] < 5).sum()),
    "sessions with unreliable done-time (flagged earlier)": int((~df["done_ok"]).sum()),
})
tab("07d_value_anomalies", anom_vals.to_frame("value"))

,value
kWh upper Tukey fence,26.96
sessions above fence,360.00
share above fence,0.03
sessions plugged in > 24 h,128.00
sessions plugged in < 15 min,276.00
arrivals between 00:00 and 05:00,332.00
sessions with unreliable done-time (flagged earlier),48.00


,value
kWh upper Tukey fence,26.96
sessions above fence,360.00
share above fence,0.03
sessions plugged in > 24 h,128.00
sessions plugged in < 15 min,276.00
arrivals between 00:00 and 05:00,332.00
sessions with unreliable done-time (flagged earlier),48.00


In [21]:
# The 13-14 kWh spike: compare its 1-kWh bin with the neighbouring bins
edges = np.arange(0, np.ceil(e.max()) + 1, 1)
hist = pd.cut(e, edges, right=False).value_counts().sort_index()
hist.index = [f"{int(i.left)}–{int(i.right)}" for i in hist.index]
tab("07e_kwh_histogram_1kwh_bins", hist.to_frame("sessions"), show=False)
display(hist.loc["9–10":"18–19"].to_frame("sessions") if "18–19" in hist.index else hist.head(20).to_frame("sessions"))

spike_n = int(e.between(13, 14, inclusive="left").sum())
neigh = [int(e.between(a, a + 1, inclusive="left").sum()) for a in (11, 12, 14, 15)]
spike_ratio = spike_n / np.mean(neigh) if np.mean(neigh) else np.nan
print(f"13–14 kWh bin: {spike_n} sessions vs {np.mean(neigh):.0f} average in the four neighbouring 1-kWh bins (ratio {spike_ratio:.1f}x)")

sp = df[e.between(13, 14, inclusive="left")]
rest = df[~e.between(13, 14, inclusive="left")]
spike_profile = pd.DataFrame({
    "spike sessions": [len(sp), sp["has_user_id"].mean(), sp["connected_h"].median(), sp["hour"].median(), sp["is_weekend"].mean()],
    "other sessions": [len(rest), rest["has_user_id"].mean(), rest["connected_h"].median(), rest["hour"].median(), rest["is_weekend"].mean()],
}, index=["sessions", "share with userID", "median hours plugged in", "median arrival hour", "share on weekends"])
tab("07f_spike_profile", spike_profile)

,sessions
9–10,559
10–11,597
11–12,640
12–13,598
13–14,2037
14–15,357
15–16,206
16–17,243
17–18,96
18–19,92


13–14 kWh bin: 2037 sessions vs 450 average in the four neighbouring 1-kWh bins (ratio 4.5x)


,spike sessions,other sessions
sessions,"2,037.00","12,242.00"
share with userID,0.04,0.13
median hours plugged in,6.56,4.39
median arrival hour,11.00,10.00
share on weekends,0.26,0.17


,spike sessions,other sessions
sessions,"2,037.00","12,242.00"
share with userID,0.04,0.13
median hours plugged in,6.56,4.39
median arrival hour,11.00,10.00
share on weekends,0.26,0.17


In [22]:
# Does the spike look like a vehicle type, a station effect, or a system artifact? Check when, where and at what power it occurs.
df["in_spike"] = e.between(13, 14, inclusive="left")
spike_month = df.groupby("month")["in_spike"].agg(spike_sessions="sum", share_of_month_sessions="mean")
tab("07g_spike_by_month", spike_month)

st_sp = df.groupby("stationID")["in_spike"].agg(spike_sessions="sum", sessions="count", spike_share="mean")
st_sp = st_sp[st_sp["sessions"] >= 50].sort_values("spike_share", ascending=False)
tab("07h_spike_share_by_station", st_sp.head(10), show=True)
print(f"Spike share across stations with >=50 sessions: min {st_sp['spike_share'].min():.1%}, median {st_sp['spike_share'].median():.1%}, max {st_sp['spike_share'].max():.1%}")

tab("07i_spike_avg_power", df.loc[df["in_spike"], "avg_power_kW"].describe().to_frame("avg_power_kW (spike sessions)"))

,spike_sessions,share_of_month_sessions
month,,
2018-04,27,0.10
2018-05,171,0.10
2018-06,153,0.08
2018-07,227,0.11
2018-08,512,0.21
2018-09,492,0.21
2018-10,407,0.17
2018-11,48,0.04


,spike_sessions,sessions,spike_share
stationID,,,
2-39-138-566,38,150,0.25
2-39-123-557,44,188,0.23
2-39-130-564,27,122,0.22
2-39-123-23,87,429,0.20
2-39-128-18,75,378,0.20
2-39-129-563,16,81,0.20
2-39-139-567,18,92,0.20
2-39-127-19,97,499,0.19
2-39-95-444,37,191,0.19


Spike share across stations with >=50 sessions: min 6.7%, median 14.5%, max 25.3%


,avg_power_kW (spike sessions)
count,"2,034.00"
mean,3.73
std,1.86
min,0.44
25%,2.10
50%,3.34
75%,5.74
max,8.61


,avg_power_kW (spike sessions)
count,"2,034.00"
mean,3.73
std,1.86
min,0.44
25%,2.10
50%,3.34
75%,5.74
max,8.61


> How to read it: a spike share that is similar every month and at every station, at a consistent power, points to a *vehicle or charger behaviour* (many cars cap at the same energy). A share that jumps at a particular month points to a *system or policy change*. A share concentrated at a few stations points to *location*. Report what the tables show; do not assert a cause they cannot support.


### 7e. Policy regimes: free vs. paid charging and the default energy allowance

The ACN-Data authors document two changes inside this window (Lee, Li & Low 2019, Fig. 2, Sec. 3.1.2):
- **until 31 Aug 2018** sessions without app input ("unclaimed") received a default of 42, 21 or 14 kWh over 12 h depending on the charger;
- **from 1 Sep 2018** the unclaimed default was 14 kWh over 12 h;
- **from 1 Nov 2018** claimed sessions paid $0.12/kWh and unclaimed sessions were stopped after 30 minutes.

If the 13–14 kWh spike is the 14 kWh default, it should be concentrated in sessions **without** a `userID`, rise after 1 Sep, and largely disappear after 1 Nov. Paid charging should show as a drop in daily volume and a jump in the share of claimed sessions. The tables below test these expectations; they do not assume them.


In [23]:

is_sp = df["kWhDelivered"].between(13, 14, inclusive="left")
days_by_regime = pd.Series(daily_counts.index, index=daily_counts.index).groupby(
    np.select([daily_counts.index < POLICY_DATES[0], daily_counts.index < POLICY_DATES[1]], REGIMES[:2], default=REGIMES[2])).size()
wd_days_by_regime = pd.Series(1, index=daily_counts.index[daily_counts.index.dayofweek < 5]).groupby(
    np.select([daily_counts.index[daily_counts.index.dayofweek < 5] < POLICY_DATES[0],
               daily_counts.index[daily_counts.index.dayofweek < 5] < POLICY_DATES[1]], REGIMES[:2], default=REGIMES[2])).sum()

reg = df.groupby("regime", observed=False).agg(sessions=("_id", "count"), median_kwh=("kWhDelivered", "median"),
                                               median_hours=("connected_h", "median"),
                                               share_with_userID=("has_user_id", "mean"))
reg["days_with_data"] = days_by_regime.reindex(REGIMES).values
reg["weekday_sessions_per_day"] = (df[~df.is_weekend].groupby("regime", observed=False).size() / wd_days_by_regime.reindex(REGIMES)).values
reg["spike_share_all"] = df.assign(s=is_sp).groupby("regime", observed=False)["s"].mean()
reg["spike_share_unclaimed"] = df[~df["has_user_id"].astype(bool)].assign(s=is_sp).groupby("regime", observed=False)["s"].mean()
reg["spike_share_claimed"] = df[df["has_user_id"].astype(bool)].assign(s=is_sp).groupby("regime", observed=False)["s"].mean()
reg["median_kwh_unclaimed"] = df[~df["has_user_id"].astype(bool)].groupby("regime", observed=False)["kWhDelivered"].median()
reg["weekday_share_arrivals_17_20h"] = df[~df.is_weekend].assign(ev=lambda d: d["hour"].between(17, 19)).groupby("regime", observed=False)["ev"].mean()
tab("07k_policy_regimes", reg)

wd_counts = daily_counts[dow_of_days < 5]
pre = wd_counts[(wd_counts.index >= POLICY_DATES[0]) & (wd_counts.index < POLICY_DATES[1])]
post = wd_counts[wd_counts.index >= POLICY_DATES[1]]
if len(post) > 5:
    paid_test = compare(post, pre, "paid", "free_R2")
    tab("07l_test_weekday_volume_paid_vs_free", paid_test.to_frame("value"))
    print(f"Weekday sessions/day: free (R2) median {pre.median():.0f} vs paid (R3) median {post.median():.0f} "
          f"({post.median()/pre.median()-1:+.0%}); effect {effect_label(paid_test.get('effect_size_rb', np.nan))}")
else:
    paid_test = pd.Series(dtype=float)
    print("Not enough weekdays after 1 Nov 2018 to compare free and paid charging.")


,sessions,median_kwh,median_hours,share_with_userID,days_with_data,weekday_sessions_per_day,spike_share_all,spike_share_unclaimed,spike_share_claimed,median_kwh_unclaimed,weekday_share_arrivals_17_20h
regime,,,,,,,,,,,
"R1 free, charger-specific defaults (to 31 Aug 2018)",8361,7.45,4.78,0.01,129,73.97,0.13,0.13,0.02,7.39,0.16
"R2 free, 14 kWh default (1 Sep - 31 Oct 2018)",4735,7.92,4.34,0.14,61,88.51,0.19,0.21,0.06,7.23,0.19
"R3 paid $0.12/kWh, unclaimed stopped after 30 min (from 1 Nov 2018)",1183,6.34,5.91,0.77,28,49.90,0.04,0.00,0.05,0.89,0.08


,value
n_paid,20.00
n_free_R2,43.00
median_paid,54.00
median_free_R2,90.00
mean_paid,49.90
mean_free_R2,88.51
p_value,0.00
effect_size_rb,-0.96


Weekday sessions/day: free (R2) median 90 vs paid (R3) median 54 (-40%); effect large


In [24]:

# Before 1 Sep the default depended on the charger (42, 21 or 14 kWh). Unclaimed sessions in R1 should then
# pile up near those values, and each charger's spike share should differ much more in R1 than in R2.
r1_un = df[(df["regime"] == REGIMES[0]) & ~df["has_user_id"].astype(bool)]["kWhDelivered"]
def bin_vs_neighbours(s, lo):
    n = int(s.between(lo, lo + 1, inclusive="left").sum())
    nb_ = np.mean([s.between(a, a + 1, inclusive="left").sum() for a in (lo - 2, lo - 1, lo + 1, lo + 2)])
    return n, nb_, n / nb_ if nb_ else np.nan
peaks = pd.DataFrame([dict(zip(["bin", "sessions", "neighbour_avg", "ratio"], (f"{lo}-{lo+1} kWh", *bin_vs_neighbours(r1_un, lo))))
                      for lo in (13, 20, 41)]).set_index("bin")
tab("07m_r1_unclaimed_default_bins", peaks)

st_reg = (df[df["regime"].isin(REGIMES[:2])].assign(s=is_sp)
            .groupby(["stationID", "regime"], observed=True)["s"].agg(["mean", "size"]).reset_index())
st_reg = st_reg[st_reg["size"] >= 30]
spread = st_reg.groupby("regime", observed=True)["mean"].agg(stations="size", min="min", median="median", max="max", std="std")
tab("07n_station_spike_share_spread_by_regime", spread)


,sessions,neighbour_avg,ratio
bin,,,
13-14 kWh,1088,266.00,4.09
20-21 kWh,33,41.00,0.80
41-42 kWh,20,8.75,2.29


,stations,min,median,max,std
regime,,,,,
"R1 free, charger-specific defaults (to 31 Aug 2018)",51,0.04,0.13,0.25,0.06
"R2 free, 14 kWh default (1 Sep - 31 Oct 2018)",51,0.05,0.19,0.36,0.07


,stations,min,median,max,std
regime,,,,,
"R1 free, charger-specific defaults (to 31 Aug 2018)",51,0.04,0.13,0.25,0.06
"R2 free, 14 kWh default (1 Sep - 31 Oct 2018)",51,0.05,0.19,0.36,0.07


> Read 7k–7n together. The documented policies are a candidate explanation; call the spike *consistent with* the default allowance only if the tables show it (concentrated in unclaimed sessions, present after 1 Sep, much smaller after 1 Nov, and more station-specific before 1 Sep). Anything modelled on the pooled data (Issue #11 onward) should either use one regime or include the regime as a feature.

## 8. Key findings
The cell below computes the headline numbers from the tables above and writes `eda_key_findings.md`.

In [25]:
wd_row, we_row = by_type.loc["Weekdays"], by_type.loc["Weekends"]
blk = blk_tab
busiest_block = blk["sessions_per_day_per_block_hour"].idxmax()     # per hour, because blocks differ in length
long_share = (df["connected_h"] >= SHORT_STAY_H).mean()
best_dow = dow["sessions_per_day"].idxmax(); worst_dow = dow["sessions_per_day"].idxmin()

findings = [
    f"**Scope.** {len(df):,} sessions at {df['stationID'].nunique()} stations, {all_days.min().date()} to {all_days.max().date()} ({n_days} calendar days, of which {n_service_days} have data), one site (open to the public; mostly campus users, so a hybrid of workplace and public charging). No annual seasonality can be assessed.",
    "**Data gaps.** " + ("; ".join(f"{r.gap_start.date()} to {r.gap_end.date()} ({int(r.days)} days with no sessions)" for r in gap_runs.itertuples()) if len(gap_runs) else "none of 7+ consecutive zero-session days") + ". Gap days are excluded from daily statistics, per-day rates, the plugged-in profile and trend tests.",
    f"**Frequency.** {daily_counts.mean():.1f} sessions per day on average (median {daily_counts.median():.0f}); weekdays {wd_row['mean']:.1f} vs weekends {we_row['mean']:.1f} per day. Dispersion index (1 = random arrivals): {by_type.loc['Weekdays','dispersion_index']:.1f} on weekdays, {by_type.loc['Weekends','dispersion_index']:.1f} on weekends (the all-days value, {by_type.loc['All days','dispersion_index']:.1f}, is inflated by mixing weekdays and weekends).",
    f"**Day of week.** Busiest {best_dow} ({dow.loc[best_dow,'sessions_per_day']:.1f} sessions/day), quietest {worst_dow} ({dow.loc[worst_dow,'sessions_per_day']:.1f}).",
    f"**Energy.** Median {df['kWhDelivered'].median():.1f} kWh per session (mean {df['kWhDelivered'].mean():.1f}, skew {df['kWhDelivered'].skew():.1f}). The largest 10% of sessions deliver {conc.set_index('share_of_sessions').loc[0.10,'share_of_energy']:.0%} of all energy. Weekday-vs-weekend energy per session: {effect_label(wk_kwh['effect_size_rb']) if 'effect_size_rb' in wk_kwh else 'n/a'} effect.",
    f"**Time of day.** {busiest_block} is the busiest block per hour ({blk.loc[busiest_block,'share_of_sessions']:.0%} of sessions, {blk.loc[busiest_block,'share_of_energy']:.0%} of energy, {blk.loc[busiest_block,'sessions_per_day_per_block_hour']:.1f} arrivals per hour per day on days with data). Weekday plugged-in vehicles peak at {wd_mean.max():.1f} around {int(wd_mean.idxmax())}:00 (peak-to-average {peak_to_avg:.2f}); the largest simultaneous count was {int(plugged.max())} of {N_STATIONS} chargers ({plugged.max()/N_STATIONS:.0%}); {cap['share of weekdays with at least one such slot']:.0%} of weekdays reach ≥90% occupancy at least once (> 72 h sessions included for occupancy).",
    f"**Duration.** Median {df['connected_h'].median():.1f} h plugged in; {long_share:.0%} of sessions last {SHORT_STAY_H:g} h or more (split at the density valley between the short- and long-stay modes, ≈{valley_h:.1f} h). Idle time is {dur_tot['idle share of plugged-in hours']:.0%} of plugged-in hours (sessions with a trusted done-time).",
    f"**Stations.** Usage Gini {gini(stn['sessions']):.2f}; busiest {top10} stations carry {share.iloc[top10-1]:.0%} of sessions (an even split would be {top10/len(stn):.0%}). Per observed day, {len(under)} stations are under-used and {len(over)} over-used by the Tukey rule.",
    f"**Anomalous days.** {len(flag)} days deviate strongly from their same-weekday baseline; {int(flag['us_federal_holiday'].sum())} fall on US federal holidays, and {int(flag['after_paid_start'].sum())} fall on or after 1 Nov 2018, where the ±2-week baseline still includes free-charging weeks. Those reflect the policy change (and Thanksgiving) rather than unusual demand.",
    f"**13–14 kWh spike.** {spike_n:,} sessions in that 1-kWh bin, {spike_ratio:.1f}x the average of the four neighbouring bins; its monthly share ranges {spike_month['share_of_month_sessions'].min():.0%}–{spike_month['share_of_month_sessions'].max():.0%} and the per-station share {st_sp['spike_share'].min():.0%}–{st_sp['spike_share'].max():.0%}. Cause not established on this evidence alone; see the regime finding.",
]
r = reg
spike_txt = (f"spike share is {r['spike_share_unclaimed'].iloc[1]:.0%} of unclaimed vs {r['spike_share_claimed'].iloc[1]:.0%} of claimed sessions in R2, "
             f"and {r['spike_share_all'].iloc[2]:.0%} of all sessions in R3 (vs {r['spike_share_all'].iloc[1]:.0%} in R2)")
consistent = (r["spike_share_unclaimed"].iloc[1] > 2 * r["spike_share_claimed"].iloc[1]) and (r["spike_share_all"].iloc[2] < 0.5 * r["spike_share_all"].iloc[1])
findings.append(
    f"**Policy regimes.** Weekday sessions per day: {r['weekday_sessions_per_day'].iloc[0]:.1f} (R1), {r['weekday_sessions_per_day'].iloc[1]:.1f} (R2), "
    f"{r['weekday_sessions_per_day'].iloc[2]:.1f} (R3, paid); share of sessions with a `userID`: {r['share_with_userID'].iloc[0]:.0%}, "
    f"{r['share_with_userID'].iloc[1]:.0%}, {r['share_with_userID'].iloc[2]:.0%}. "
    + (f"Paid vs free (R2) weekday volume: {effect_label(paid_test['effect_size_rb'])} effect. " if 'effect_size_rb' in paid_test else "")
    + f"For the 13–14 kWh spike, {spike_txt}"
    + ("; this is consistent with the 14 kWh default allowance for unclaimed sessions documented by the dataset authors." if consistent
       else "; this does not clearly match the documented default allowance, so the cause stays open.")
    + f" R3 covers only {int(r['days_with_data'].iloc[2])} days with data, so paid-period figures are indicative.")

# Evidence detail for the default allowance: what the data confirms after 1 Nov, and whether the
# charger-specific 21/42 kWh defaults described for the period before 1 Sep are visible.
r21, r42 = peaks.loc["20-21 kWh", "ratio"], peaks.loc["41-42 kWh", "ratio"]
n42 = int(peaks.loc["41-42 kWh", "sessions"])
sd_r1, sd_r2 = spread["std"].iloc[0], spread["std"].iloc[1]
r1_visible = (r21 > 1.5 or r42 > 1.5) and sd_r1 > sd_r2
findings.append(
    f"**Default allowance, evidence detail.** After 1 Nov the spike share among unclaimed sessions is {r['spike_share_unclaimed'].iloc[2]:.0%} "
    f"and their median energy {r['median_kwh_unclaimed'].iloc[2]:.2f} kWh, consistent with unclaimed sessions being stopped after 30 minutes. "
    f"Before 1 Sep, unclaimed sessions show a ratio of {r21:.1f}x at 20–21 kWh and {r42:.1f}x at 41–42 kWh ({n42} sessions) versus neighbouring bins, and the "
    f"station-level spike share spread is {sd_r1:.2f} (R1) vs {sd_r2:.2f} (R2), so the charger-specific 21/42 kWh defaults described for that period are "
    + ("visible in this data." if r1_visible else "not clearly visible in this data; that part of the documented policy is not confirmed here."))

if HAS_SCIPY and "sessions_per_station_per_service_day" in trend:
    t = trend["sessions_per_station_per_service_day"]
    col = "sessions_per_station_per_service_day"
    first_m, last_m = full.iloc[0], full.iloc[-1]
    peak_m = full[col].idxmax()
    if peak_m not in (full.index[0], full.index[-1]):
        shape = (f"rose from {first_m[col]:.2f} ({full.index[0]}) to a peak of {full.loc[peak_m, col]:.2f} ({peak_m}), then fell to "
                 f"{last_m[col]:.2f} ({full.index[-1]}); a straight-line trend statistic hides this rise-then-fall shape")
    else:
        shape = f"went from {first_m[col]:.2f} ({full.index[0]}) to {last_m[col]:.2f} ({full.index[-1]})"
    findings.append(f"**Trend.** Sessions per active station per day with data {shape}. Active stations changed only from {int(first_m['active_stations'])} to {int(last_m['active_stations'])}, so rollout does not explain it. Spearman rho {t['spearman_rho_vs_time']:.2f} (p={t['p_value']:.2f}) over {len(full)} months, all in the free-charging period: not a firm conclusion. Cause not established.")

text = "# EDA key findings (numbers computed from the data)\n\n" + "\n\n".join(f"- {f}" for f in findings) + "\n"
with open("eda_key_findings.md", "w", encoding="utf-8") as fh:
    fh.write(text)
display(Markdown(text))

# EDA key findings (numbers computed from the data)

- **Scope.** 14,279 sessions at 54 stations, 2018-04-25 to 2018-11-28 (218 calendar days, of which 218 have data), one site (open to the public; mostly campus users, so a hybrid of workplace and public charging). No annual seasonality can be assessed.

- **Data gaps.** none of 7+ consecutive zero-session days. Gap days are excluded from daily statistics, per-day rates, the plugged-in profile and trend tests.

- **Frequency.** 65.5 sessions per day on average (median 69); weekdays 74.9 vs weekends 41.9 per day. Dispersion index (1 = random arrivals): 4.0 on weekdays, 4.1 on weekends (the all-days value, 7.4, is inflated by mixing weekdays and weekends).

- **Day of week.** Busiest Tue (77.5 sessions/day), quietest Sun (41.5).

- **Energy.** Median 7.5 kWh per session (mean 9.0, skew 2.1). The largest 10% of sessions deliver 26% of all energy. Weekday-vs-weekend energy per session: negligible effect.

- **Time of day.** Morning 06–10 is the busiest block per hour (42% of sessions, 43% of energy, 6.9 arrivals per hour per day on days with data). Weekday plugged-in vehicles peak at 38.5 around 11:00 (peak-to-average 2.03); the largest simultaneous count was 51 of 54 chargers (94%); 3% of weekdays reach ≥90% occupancy at least once (> 72 h sessions included for occupancy).

- **Duration.** Median 4.7 h plugged in; 43% of sessions last 6 h or more (split at the density valley between the short- and long-stay modes, ≈6.1 h). Idle time is 44% of plugged-in hours (sessions with a trusted done-time).

- **Stations.** Usage Gini 0.31; busiest 10 stations carry 34% of sessions (an even split would be 19%). Per observed day, 0 stations are under-used and 1 over-used by the Tukey rule.

- **Anomalous days.** 25 days deviate strongly from their same-weekday baseline; 4 fall on US federal holidays, and 9 fall on or after 1 Nov 2018, where the ±2-week baseline still includes free-charging weeks. Those reflect the policy change (and Thanksgiving) rather than unusual demand.

- **13–14 kWh spike.** 2,037 sessions in that 1-kWh bin, 4.5x the average of the four neighbouring bins; its monthly share ranges 4%–21% and the per-station share 7%–25%. Cause not established on this evidence alone; see the regime finding.

- **Policy regimes.** Weekday sessions per day: 74.0 (R1), 88.5 (R2), 49.9 (R3, paid); share of sessions with a `userID`: 1%, 14%, 77%. Paid vs free (R2) weekday volume: large effect. For the 13–14 kWh spike, spike share is 21% of unclaimed vs 6% of claimed sessions in R2, and 4% of all sessions in R3 (vs 19% in R2); this is consistent with the 14 kWh default allowance for unclaimed sessions documented by the dataset authors. R3 covers only 28 days with data, so paid-period figures are indicative.

- **Default allowance, evidence detail.** After 1 Nov the spike share among unclaimed sessions is 0% and their median energy 0.89 kWh, consistent with unclaimed sessions being stopped after 30 minutes. Before 1 Sep, unclaimed sessions show a ratio of 0.8x at 20–21 kWh and 2.3x at 41–42 kWh (20 sessions) versus neighbouring bins, and the station-level spike share spread is 0.06 (R1) vs 0.07 (R2), so the charger-specific 21/42 kWh defaults described for that period are not clearly visible in this data; that part of the documented policy is not confirmed here.

- **Trend.** Sessions per active station per day with data rose from 1.12 (2018-05) to a peak of 1.48 (2018-08), then fell to 1.45 (2018-10); a straight-line trend statistic hides this rise-then-fall shape. Active stations changed only from 51 to 54, so rollout does not explain it. Spearman rho 0.83 (p=0.04) over 6 months, all in the free-charging period: not a firm conclusion. Cause not established.


## Issue #10 checklist → where it is done

| Task | Section |
|---|---|
| Analyze charging-session frequency | 1 |
| Analyze energy consumption patterns | 2 |
| Analyze charging duration | 3 |
| Analyze demand by time of day | 4 |
| Analyze demand by day/week patterns | 5 |
| Analyze station/site-level usage | 6 |
| Identify important trends and anomalies | 7 (incl. 7e policy regimes) |

**Deliverables:** this notebook (with outputs saved), the `eda_tables/` folder (statistical summaries), and `eda_key_findings.md`.